# Centralized Model Interpretability & Explainability (Incongruent Mirrored vs. Incongruent Pure)

This notebook centralizes feature attribution for the deep models. It should be read as exploratory: the cross-model results make Riemannian TS-LDA the primary benchmark, while these plots help inspect what the neural models learned. It contains:
1. **Backpropagation-based Saliency Mapping** (spatial scalp topographies and temporal gradient profiles).
2. **SHAP (SHapley Additive exPlanations)** feature attribution analysis using `shap.GradientExplainer`.

We compare three advanced neural network architectures:
- **ST-GNN**: Spatio-Temporal Graph Neural Network using Chebyshev convolutions over the sensor adjacency graph.
- **ST-EEGFormer**: Spatio-Temporal EEG Transformer with spatial and temporal self-attention.
- **LaBraM**: Large Brain Model pretrained on massive EEG datasets and adapted via LoRA.

In [ ]:
# Unified Environment Setup (Local, Colab, Kaggle)
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    print('Detected Kaggle environment. Installing dependencies...')
    !pip install -q mne braindecode[hub] safetensors timm shap huggingface_hub
elif IS_COLAB:
    print('Detected Google Colab environment. Installing dependencies...')
    !pip install -q mne braindecode[hub] safetensors timm shap huggingface_hub
    from google.colab import drive
    drive.mount('/content/drive')

# Locate workspace source directory
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

if IS_KAGGLE:
    # Dynamically find the unzipped dataset path
    dataset_root = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'tfg-nils-2026-dataset.zip' in files or 'br_analysis' in dirs:
            dataset_root = Path(root)
            break
    SRC_DIR = dataset_root if dataset_root else Path('/kaggle/input/tfg-nils-2026-dataset')
elif IS_COLAB:
    SRC_DIR = Path('/content/drive/MyDrive/TFG_Nils_2026/03_Codi')
else:
    SRC_DIR = NOTEBOOK_DIR.parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Using source directory: {SRC_DIR}')


In [ ]:
try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception:
    pass

import mne
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from br_analysis import detect_project_paths, get_task_config, load_raw_epoch_dataset, RawEpochSpec
from br_analysis.torch_training import build_dataloader, standardize_train_test, chronological_group_labels, stratified_group_splits

TASK_NAME = 'Incongruent Mirrored vs. Incongruent Pure'
TASK_TAG = 'im_vs_ip'
paths = detect_project_paths('explainability_im_vs_ip')

mne.set_log_level('WARNING')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Working Paths:')
print('  Cache:', paths.cache_dir)
print('  Tables:', paths.tables_dir)
print('  Figures:', paths.figures_dir)


## 1. Load Preprocessed Data
We load the raw EEG epochs resampled to 128 Hz over the 1.5s post-stimulus interval.

In [ ]:
# Load preprocessed epochs
task = get_task_config(TASK_NAME, paths.analysis_root)
spec = RawEpochSpec(bandpass=(1.0, 40.0), resample_sfreq=128.0, analysis_window=(0.0, 1.5), peak_align_all_channels=True)
epochs = load_raw_epoch_dataset(paths, task, spec)

X = epochs.get_data(units='V')  # (trials, channels, times)
y = epochs.labels
subjects = epochs.subjects
ch_names = list(epochs.channel_names)
TARGET_LEN = X.shape[-1]
sfreq = 128.0

SUBJECT_ID = 'P057'  # Selected subject for detailed explainability
print(f'Total Trials: {X.shape}, Classes: {np.bincount(y)}')


## 2. Saliency Mapping Comparison

Saliency maps are computed by calculating the gradient of the model's output w.r.t the input EEG samples. The magnitude of these gradients indicates how sensitive the model's prediction is to changes at specific sensors (spatial) and time points (temporal).

In [ ]:
# Define model architectures and loading helpers
import torch.nn as nn
from br_analysis.torch_models import STGNNChebConv
from br_analysis import mae_vit_small_patch16, temporal_band_features, electrode_distance_adjacency
from braindecode.models import InterpolatedLaBraM
from br_analysis.pretrained import prepare_lora_backbone, BackboneClassifier

device = torch.device(DEVICE)
EXPLAIN_FOLD = 1  # representative held-out fold for local attribution diagnostics
MODEL_PATHS = {
    "stgnn": detect_project_paths("im_vs_i_stgnn"),
    "steegformer": detect_project_paths("im_vs_i_steegformer"),
    "labram": detect_project_paths("im_vs_i_labram"),
}
MODEL_TAGS = {
    "stgnn": "im_vs_ip_stgnn",
    "steegformer": "im_vs_ip_steegformer",
    "labram": "im_vs_ip_labram",
}

X_sub = X[subjects == SUBJECT_ID]
y_sub = y[subjects == SUBJECT_ID]
splits = list(stratified_group_splits(y_sub, groups=chronological_group_labels(len(y_sub)), n_splits=5))
train_idx, test_idx = splits[EXPLAIN_FOLD - 1]
X_train, X_test = standardize_train_test(X_sub[train_idx], X_sub[test_idx])
y_test = y_sub[test_idx]
conflict_trials = X_test[y_test == 1]

# ST-GNN was trained on graph/time-window features, not raw epochs.
graph_bands = {"theta": (5.0, 7.0), "alpha": (8.0, 12.0)}
X_graph, window_centers, graph_band_names = temporal_band_features(
    X,
    sfreq,
    bands=graph_bands,
    window_sec=0.500,
    step_sec=0.100,
    mode="of",
)
X_graph_sub = X_graph[subjects == SUBJECT_ID].astype(np.float32)
X_graph_train, X_graph_test = standardize_train_test(X_graph_sub[train_idx], X_graph_sub[test_idx])
conflict_graph_trials = X_graph_test[y_test == 1]
adjacency = electrode_distance_adjacency(ch_names, threshold=0.35)

def _load_state_if_available(model, path, *, strict=True):
    if not path.exists():
        print(f"[WARN] Missing weights: {path}")
        return False
    model.load_state_dict(torch.load(path, map_location=device), strict=strict)
    return True

def load_stgnn():
    model = STGNNChebConv(n_nodes=len(ch_names), in_features=len(graph_band_names), adjacency=adjacency, n_classes=2)
    path = MODEL_PATHS["stgnn"].cache_dir / f"{MODEL_TAGS['stgnn']}_sub{SUBJECT_ID}_fold{EXPLAIN_FOLD}.pt"
    _load_state_if_available(model, path)
    return model.to(device).eval()

def load_steegformer():
    backbone = mae_vit_small_patch16()
    prepare_lora_backbone(backbone, include_substrings=("attn", "mlp"), r=8, alpha=16.0)
    model = BackboneClassifier(backbone, n_classes=2)
    path = MODEL_PATHS["steegformer"].cache_dir / f"{MODEL_TAGS['steegformer']}_sub{SUBJECT_ID}_fold{EXPLAIN_FOLD}.pt"
    _load_state_if_available(model, path, strict=False)
    return model.to(device).eval()

def load_labram():
    montage = mne.channels.make_standard_montage("standard_1005")
    info = mne.create_info(ch_names=ch_names, sfreq=128.0, ch_types="eeg")
    info.set_montage(montage, on_missing="ignore")
    model = InterpolatedLaBraM(chs_info=info["chs"], n_times=200, n_outputs=2, patch_size=200)
    prepare_lora_backbone(
        model,
        include_substrings=("attn", "mlp"),
        r=8,
        alpha=16.0,
        exclude_substrings=("head", "classifier", "interpolation_layer"),
    )
    path = MODEL_PATHS["labram"].cache_dir / f"{MODEL_TAGS['labram']}_sub{SUBJECT_ID}_fold{EXPLAIN_FOLD}.pt"
    _load_state_if_available(model, path, strict=False)
    return model.to(device).eval()

print('Model loaders ready.')


In [ ]:
# Compute Saliency Maps
def compute_saliency(model, x_data, *, is_transformer=False, is_labram=False):
    if is_transformer:
        from br_analysis.pretrained import align_channels_to_steegformer
        from br_analysis.config import ST_EEGFORMER_CHANNELS
        x_in = align_channels_to_steegformer(x_data, ch_names)
        X_tensor = torch.from_numpy(x_in).float().to(device)
    elif is_labram:
        pad_width = max(0, 200 - x_data.shape[-1])
        x_in = np.pad(x_data, ((0, 0), (0, 0), (0, pad_width)), mode="edge")
        X_tensor = torch.from_numpy(x_in[:, :, :200]).float().to(device)
    else:
        X_tensor = torch.from_numpy(x_data).float().to(device)

    X_tensor.requires_grad = True
    out = model(X_tensor)
    loss = out[:, 1].sum()
    loss.backward()
    grad = X_tensor.grad.abs().cpu().numpy().mean(axis=0)

    if is_transformer:
        from br_analysis.config import ST_EEGFORMER_CHANNELS
        target_index_by_name = {name.upper(): idx for idx, name in enumerate(ST_EEGFORMER_CHANNELS)}
        original_indices = [target_index_by_name[ch.upper()] for ch in ch_names]
        grad = grad[original_indices, :TARGET_LEN]
    elif is_labram:
        grad = grad[:, :TARGET_LEN]
    return grad

try:
    stgnn = load_stgnn()
    steeg = load_steegformer()
    labram = load_labram()

    sal_stgnn = compute_saliency(stgnn, conflict_graph_trials)
    sal_steeg = compute_saliency(steeg, conflict_trials, is_transformer=True)
    sal_labram = compute_saliency(labram, conflict_trials, is_labram=True)

    stgnn_spatial = sal_stgnn.mean(axis=(0, 2))
    steeg_spatial = sal_steeg.mean(axis=1)
    labram_spatial = sal_labram.mean(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    montage = mne.channels.make_standard_montage("standard_1005")
    info = mne.create_info(ch_names=ch_names, sfreq=128.0, ch_types="eeg")
    info.set_montage(montage, on_missing="ignore")

    models_sal = [("ST-GNN Saliency", stgnn_spatial), ("ST-EEGFormer Saliency", steeg_spatial), ("LaBraM Saliency", labram_spatial)]
    for idx, (title, spatial) in enumerate(models_sal):
        im, _ = mne.viz.plot_topomap(spatial, info, axes=axes[idx], cmap="hot", show=False)
        axes[idx].set_title(title)
        fig.colorbar(im, ax=axes[idx], shrink=0.6)

    plt.tight_layout()
    plt.savefig(paths.figures_dir / f"saliency_spatial_comparison_{TASK_TAG}.png", dpi=180)
    plt.show()

    t_sec = np.arange(TARGET_LEN) / sfreq
    plt.figure(figsize=(10, 4))
    plt.plot(window_centers, sal_stgnn.mean(axis=(1, 2)), label="ST-GNN", color="green")
    plt.plot(t_sec, sal_steeg.mean(axis=0), label="ST-EEGFormer", color="blue")
    plt.plot(t_sec, sal_labram.mean(axis=0), label="LaBraM", color="red")
    plt.title(f"Temporal Saliency Sensitivity Profiles Comparison ({SUBJECT_ID})")
    plt.xlabel("Time post-stimulus (s)")
    plt.ylabel("Mean gradient magnitude")
    plt.legend()
    plt.grid(True, linestyle=":")
    plt.savefig(paths.figures_dir / f"saliency_temporal_comparison_{TASK_TAG}.png", dpi=180)
    plt.show()
except Exception as e:
    print("Error computing/plotting Saliency. Check that model weights exist for the selected representative fold:", e)


## 3. SHAP Feature Attributions Comparison

SHAP value attributions show how much each sensor/time sample contributed to push the model's output prediction away from its baseline/background expected prediction. We use `shap.GradientExplainer` with a representative background dataset and a conflict trials test set (optimized to explain the first 10 trials for speed).

In [ ]:
import shap

def compute_shap(model, x_data, bg_data, *, is_transformer=False, is_labram=False):
    if is_transformer:
        from br_analysis.pretrained import align_channels_to_steegformer
        from br_analysis.config import ST_EEGFORMER_CHANNELS
        x_in = align_channels_to_steegformer(x_data, ch_names)
        bg_in = align_channels_to_steegformer(bg_data, ch_names)
        bg_tensor = torch.from_numpy(bg_in[:30]).float().to(device)
        test_tensor = torch.from_numpy(x_in[:10]).float().to(device)
    elif is_labram:
        pad_width_x = max(0, 200 - x_data.shape[-1])
        pad_width_bg = max(0, 200 - bg_data.shape[-1])
        x_in = np.pad(x_data, ((0, 0), (0, 0), (0, pad_width_x)), mode="edge")[:, :, :200]
        bg_in = np.pad(bg_data, ((0, 0), (0, 0), (0, pad_width_bg)), mode="edge")[:, :, :200]
        bg_tensor = torch.from_numpy(bg_in[:30]).float().to(device)
        test_tensor = torch.from_numpy(x_in[:10]).float().to(device)
    else:
        bg_tensor = torch.from_numpy(bg_data[:30]).float().to(device)
        test_tensor = torch.from_numpy(x_data[:10]).float().to(device)

    explainer = shap.GradientExplainer(model, bg_tensor)
    shap_vals = explainer.shap_values(test_tensor)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    elif isinstance(shap_vals, np.ndarray) and shap_vals.shape[-1] == 2:
        shap_vals = shap_vals[..., 1]

    if is_transformer:
        from br_analysis.config import ST_EEGFORMER_CHANNELS
        target_index_by_name = {name.upper(): idx for idx, name in enumerate(ST_EEGFORMER_CHANNELS)}
        original_indices = [target_index_by_name[ch.upper()] for ch in ch_names]
        shap_vals = shap_vals[:, original_indices, :TARGET_LEN]
    elif is_labram:
        shap_vals = shap_vals[:, :, :TARGET_LEN]
    return np.abs(shap_vals).mean(axis=0)

try:
    print("Computing SHAP values for ST-GNN...")
    shap_stgnn = compute_shap(stgnn, conflict_graph_trials, X_graph_test)

    print("Computing SHAP values for ST-EEGFormer...")
    shap_steeg = compute_shap(steeg, conflict_trials, X_test, is_transformer=True)

    print("Computing SHAP values for LaBraM...")
    shap_labram = compute_shap(labram, conflict_trials, X_test, is_labram=True)

    stgnn_spatial = shap_stgnn.mean(axis=(0, 2))
    steeg_spatial = shap_steeg.mean(axis=1)
    labram_spatial = shap_labram.mean(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    models_shap = [("ST-GNN SHAP", stgnn_spatial), ("ST-EEGFormer SHAP", steeg_spatial), ("LaBraM SHAP", labram_spatial)]
    for idx, (title, spatial) in enumerate(models_shap):
        im, _ = mne.viz.plot_topomap(spatial, info, axes=axes[idx], cmap="viridis", show=False)
        axes[idx].set_title(title)
        fig.colorbar(im, ax=axes[idx], shrink=0.6, label="Mean SHAP magnitude")

    plt.tight_layout()
    plt.savefig(paths.figures_dir / f"shap_spatial_comparison_{TASK_TAG}.png", dpi=180)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(window_centers, shap_stgnn.mean(axis=(1, 2)), label="ST-GNN", color="green")
    plt.plot(t_sec, shap_steeg.mean(axis=0), label="ST-EEGFormer", color="blue")
    plt.plot(t_sec, shap_labram.mean(axis=0), label="LaBraM", color="red")
    plt.title(f"Temporal SHAP Attribution Profiles Comparison ({SUBJECT_ID})")
    plt.xlabel("Time post-stimulus (s)")
    plt.ylabel("Mean SHAP magnitude")
    plt.legend()
    plt.grid(True, linestyle=":")
    plt.savefig(paths.figures_dir / f"shap_temporal_comparison_{TASK_TAG}.png", dpi=180)
    plt.show()
except Exception as e:
    print("Error computing SHAP values:", e)


In [ ]:
# Zipping outputs for easy download when running on Kaggle
if IS_KAGGLE:
    import shutil
    print('Zipping outputs...')
    shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/TFG_Nils_2026/04_Anàlisi')
    print('Zipped outputs saved to /kaggle/working/outputs.zip')
